In [10]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

# os.environ["TRANSFORMERS_CACHE"] = "/content/drive/Shareddrives/Algoverse_KSAC/hf_cache" #stores model
os.environ["HF_HOME"] = "./hf_home"  # stores logins

hf_token = os.getenv('HF_TOKEN')
# print(type(hf_token))
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pathlib import Path

variants = ['small', 'base', 'large']

paths = {variant:f'./models/google/flan-t5-{variant}' for variant in variants}

model_var = 'large'
current_path =paths[model_var]
save_path = Path(current_path).resolve()

tokenizer = AutoTokenizer.from_pretrained(
    save_path,
    local_files_only=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    save_path,
    torch_dtype="auto",
    device_map="auto",
    local_files_only=True
)

print("Reloaded model successfully")
print(f"model.device = {model.device}")

Reloaded model successfully
model.device = mps:0


#Dataloading (TODO)
#after sparqlgen + name parsing

In [12]:
import pandas as pd

In [13]:
names = ['mintaka', 'hotpot', 'qald']

dataframes = {name:pd.read_csv(f'./processed_names/{name}_processed.csv') for name in names}





In [14]:
# from transformers import pipeline
from transformers.generation.utils import GenerationMixin

In [15]:
def answer(question):
    # Simpler, more direct prompt for FLAN
    prompt = f"""

Question: {question}

Answer the given question to the best of your knowledge. Provide a succint and clear short answer."""
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=4,
        temperature = 0,
        early_stopping=True,
        do_sample=False,
    )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    return generated_text

answered = {}
for name, data in dataframes.items():
    print(f'Starting processing for {name}')
    answers = []
    for index, row in data.iterrows():
        question = row['AAVE Question']
        llmanswer = answer(question)
        answers.append(llmanswer)
        print(f'Row {index+1} processed', llmanswer)
    data['Answer'] = answers
    answered[name] = data

Starting processing for mintaka
Row 1 processed Mount Everest
Row 2 processed Jeremy Irons
Row 3 processed Jeremy Irons
Row 4 processed 2001
Row 5 processed John Kitzhaber
Row 6 processed George W. Bush
Row 7 processed The construction of the Lincoln Memorial was completed in 1939.
Row 8 processed No
Row 9 processed Edward Cullen
Row 10 processed 17
Row 11 processed Mississippi River
Row 12 processed 20 years
Row 13 processed Hawaii
Row 14 processed Lake Michigan
Row 15 processed 0
Row 16 processed Brooklyn, New York
Row 17 processed September 11, 2001
Row 18 processed Metroid
Row 19 processed Super Mario Odyssey
Row 20 processed Texas is the largest state in the United States.
Row 21 processed 0
Row 22 processed antonio poltico
Row 23 processed Norma "Rose" Smith
Row 24 processed No, the Rams have not won a Super Bowl ring.
Row 25 processed John Kitzhaber
Row 26 processed AAVE
Row 27 processed Venus Williams is a tennis player. Serena Williams is an actress.
Row 28 processed Barry Bon

In [16]:
import ast
import re

def clean_text(text):
    # If the entry is actually a list, flatten it
    if isinstance(text, list):
        text = text[0]
    elif isinstance(text, str):
       #If looks like a list []
        if text.strip().startswith('[') and text.strip().endswith(']'):
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, list) and len(parsed) > 0:
                    text = parsed[0]
            except Exception:
                pass

    # Now clean as before
    if not isinstance(text, str):
        return text

    text = re.sub(r'^(AAVE|SAE)\s*Question[:\s-]*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^[\s\[\]\'"]+|[\s\[\]\'"]+$', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text



for name, data in answered.items():
# Apply to both columns
  data['SAE Question'] = data['SAE Question'].apply(clean_text)
  data['AAVE Question'] = data['AAVE Question'].apply(clean_text)
  file_path = f'./LLM_answers/vanilla-flan-t5-{model_var}/Vanilla_LLM_Answers_{name}.csv'
  data.to_csv(file_path, index=False)
  print(f"Translations completed and saved to {file_path}")





Translations completed and saved to ./LLM_answers/vanilla-flan-t5-base/Vanilla_LLM_Answers_mintaka.csv
Translations completed and saved to ./LLM_answers/vanilla-flan-t5-base/Vanilla_LLM_Answers_hotpot.csv
Translations completed and saved to ./LLM_answers/vanilla-flan-t5-base/Vanilla_LLM_Answers_qald.csv
